In [0]:
import os, sys
import logging
import datetime
from datetime import datetime, timedelta
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    expr,
    from_utc_timestamp,
    lit,
    max,
    sha2,
    substring,
    to_date,
    when,
    broadcast,
    trim,
)

In [0]:
from functools import wraps
import operator

def reconcile_table(table_name, env_prefix, run_date):
    current_env = spark.sql(f"SELECT COUNT(*) AS countRows FROM `{env_prefix}-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'")
    prod = spark.sql(f"SELECT COUNT(*) AS countRows FROM `p-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'")
    if spark.catalog.tableExists(f"`{env_prefix}-cdh`.landed.{table_name}"):
        count_current_env, count_prod = current_env.collect()[0]["countRows"], prod.collect()[0]["countRows"]
        if operator.ne(count_current_env, count_prod):
            print(f"Table: {table_name} | Copying Run Date data: {run_date} | prod -> {env_prefix}")
            spark.sql(f"""DELETE FROM `{env_prefix}-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'""")
            spark.sql(f"""INSERT INTO `{env_prefix}-cdh`.landed.{table_name} SELECT * FROM `p-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'""")
        else:
            print(f"Table: {table_name} | No changes | prod -> {env_prefix}")

    else:
        print(f"Table: {table_name} | Copying Prodf table | prod -> {env_prefix}")
        spark.sql(f"""CREATE OR REPLACE TABLE `{env_prefix}-cdh`.landed.{table_name} CLONE `p-cdh`.landed.{table_name}""")


def create_decorator(function):
    @wraps(function)
    def wrapper(spark, catalog_landed, table_name, table_filters, table_selected_fields, *args, **kwargs):
        nb = sys.modules["__main__"]
        env_prefix  = getattr(nb, "env_prefix", None)
        run_date    = getattr(nb, "run_date", None)

        reconcile_table(table_name, env_prefix, run_date)
        result = function(spark, catalog_landed, table_name, table_filters, table_selected_fields, *args, **kwargs)
        return result
    return wrapper


from pyspark.sql import SparkSession, DataFrame
from typing import Optional
from pyspark.sql.functions import trim

def load_landed_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    # Trim all string columns
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

def load_landed_portfolio_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    # Trim all string columns
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

def create_delivery_delta_table(
    spark: SparkSession,
    delivery_table_path: str,
    cluster_columns: Optional[list[str]],
    schema: str,
) -> None:


    sql_query = f"""
        CREATE TABLE IF NOT EXISTS {delivery_table_path} ( 
        {schema}
        )
        USING DELTA
    """
    
    if cluster_columns:
        sql_query = sql_query + f"CLUSTER BY ({', '.join(cluster_columns)})"

    sql_query = (
        sql_query
        + """
        TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.minReaderVersion' = '3',
            'delta.minWriterVersion' = '7',
            'delta.columnMapping.mode' = 'name',
            'delta.dataSkippingStatsColumns' = 'BusinessDate'
        )
    """
    )

    spark.sql(sql_query)
    spark.sql(
            f"""REPAIR TABLE {delivery_table_path} SYNC METADATA"""   
        )
    
def fetch_max_business_date(spark: SparkSession, catalog_name: str, table_name: str):
    max_business_date = spark.sql(
        f"""SELECT MAX(BusinessDate) AS maxBusinessDate FROM {catalog_name}.{table_name}"""
        ).collect()[0]['maxBusinessDate']
    return max_business_date

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('delivery_zone', 'None', '03-Delivery_zone')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
delivery_zone = dbutils.widgets.get('delivery_zone')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

In [0]:
spark = SparkSession.builder.getOrCreate()
t1 = fetch_max_business_date(spark=spark, catalog_name=catalog_landed, table_name='sme_business_plan_forc_cbs_nonre')
t2 = fetch_max_business_date(spark=spark, catalog_name=catalog_landed, table_name='sme_business_plan_forc_cbs_re_3rdparty')
t3 = fetch_max_business_date(spark=spark, catalog_name=catalog_landed, table_name='sme_business_plan_forc_cbs_reo_hold')
t4 = fetch_max_business_date(spark=spark, catalog_name=catalog_landed, table_name='sme_business_plan_forc_cbs_reo_sale')
t5 = fetch_max_business_date(spark=spark, catalog_name=curated_zone, table_name='portfolio')
run_date = min(list(filter(lambda e: e is not None, (t1, t2, t3, t4, t5))))
run_date = datetime.strftime(run_date, '%Y-%m-%d')
print(f'Run Date: {run_date} |Landed catalog: {catalog_landed} |Curated zone: {curated_zone} |Environment: {env_prefix}')

In [0]:
schema = (
    "BusinessPlanID INT, DebtorCDI String, AgreementNumber String,"
    "BusinessPlan_Status String, BusinessPlanStrategy_Code Int,"
    "BusinessPlan_SolutionType String, BusinessPlan_Comments String,"
    "BusinessPlanStrategy_Status String, Confidence String, ProposedBy String,"
    "BusinessPlanStrategy_StrategyType String, BusinessPlanStrategy_Comments String,"
    "LastModifiedDate Date, PortfolioName String, TimeToAuction Int, AuctionPrice Decimal(19,2),"
    "TimeToSell Int, NetRecoveryAmount Decimal(19, 2), BusinessDate Date"
)

cluster_columns = ["BusinessDate"]
date_filter = f"businessdate = {run_date!r}"
delivery_table_path = f"{delivery_zone}.SME_Tool_Forc_CBS" 

# ================== Create Delta Table ==================

logging.info("Creating Delta Table...")
if not spark.catalog.tableExists(delivery_table_path):
    create_delivery_delta_table(
        spark = spark,
        cluster_columns = cluster_columns,
        delivery_table_path = delivery_table_path,
        schema = schema
        ) 
else:
    logging.info("Table already exists")

# ============================== sme_business_plan_forc_cbs_nonre ====================================

nonre_extra_filters = None
if nonre_extra_filters is not None:
    nonre_extra_filters.append(date_filter)
    nonre_final_filters = " AND ".join(
        nonre_extra_filters
    )
else:
    nonre_final_filters = date_filter

nonre_selected_filters = [

    "BusinessPlanID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "TimeToRecovery",
    "NetRecoveryAmount",
    "BusinessDate"
]

nonre = load_landed_delta_table(
    spark,
    catalog_landed,  
    "sme_business_plan_forc_cbs_nonre",
    nonre_final_filters,
    nonre_selected_filters,
    )

# ==================================== sme_business_plan_forc_cbs_re_3rdparty =======================================

re_3rdparty_extra_filters = None
if re_3rdparty_extra_filters is not None:
    re_3rdparty_extra_filters.append(date_filter)
    re_3rdparty_final_filters = " AND ".join(
        re_3rdparty_extra_filters
    )
else:
    re_3rdparty_final_filters = date_filter

re_3rdparty_selected_filters = [

    "BusinessPlanID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "AuctionPrice",
    "TimeToAuction",
    "NetRecoveryAmount",
    "BusinessDate"
]

re_3rdparty = load_landed_delta_table(
    spark,
    catalog_landed,  
    "sme_business_plan_forc_cbs_re_3rdparty",
    re_3rdparty_final_filters,
    re_3rdparty_selected_filters,
    )

# ==================================== sme_business_plan_forc_cbs_reo_hold =======================================

reo_hold_extra_filters = None
if reo_hold_extra_filters is not None:
    reo_hold_extra_filters.append(date_filter)
    reo_hold_final_filters = " AND ".join(
        reo_hold_extra_filters
    )
else:
    reo_hold_final_filters = date_filter

reo_hold_selected_filters = [

    "BusinessPlanID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "AuctionPrice",
    "TimeToAuction",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

reo_hold = load_landed_delta_table(
    spark,
    catalog_landed,  
    "sme_business_plan_forc_cbs_reo_hold",
    reo_hold_final_filters,
    reo_hold_selected_filters,
    )

# ==================================== sme_business_plan_forc_cbs_reo_sale =======================================

reo_sale_extra_filters = None
if reo_sale_extra_filters is not None:
    reo_sale_extra_filters.append(date_filter)
    reo_sale_final_filters = " AND ".join(
        reo_sale_extra_filters
    )
else:
    reo_sale_final_filters = date_filter

reo_sale_selected_filters = [

    "BusinessPlanID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "AuctionPrice",
    "TimeToAuction",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

reo_sale = load_landed_delta_table(
    spark,
    catalog_landed,  
    "sme_business_plan_forc_cbs_reo_sale",
    reo_sale_final_filters,
    reo_sale_selected_filters,
    )

# ================== portfolio ==================

portfolio_extra_filters = None

if portfolio_extra_filters is not None:
    portfolio_extra_filters.append(date_filter)
    portfolio_final_filters = " AND ".join(portfolio_extra_filters)
else:
    portfolio_final_filters = date_filter 

portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_sme",
        ]

portfolio = load_landed_portfolio_table( 
            spark,
            curated_zone,  
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

# ================== Joins ==================

nonre_join_condition = (
    nonre["PortfolioID"] == portfolio["portfolio_id_sme"]
    )

re_3rdparty_join_condition = (
    re_3rdparty["PortfolioID"] == portfolio["portfolio_id_sme"]
    )

reo_hold_join_condition = (
    reo_hold["PortfolioID"] == portfolio["portfolio_id_sme"]
    )

reo_sale_join_condition = (
    reo_sale["PortfolioID"] == portfolio["portfolio_id_sme"]
    )

nonre = nonre.join(broadcast(portfolio), nonre_join_condition)
re_3rdparty = re_3rdparty.join(broadcast(portfolio), re_3rdparty_join_condition)
reo_hold = reo_hold.join(broadcast(portfolio), reo_hold_join_condition)
reo_sale = reo_sale.join(broadcast(portfolio), reo_sale_join_condition)

# ================== Target tables schema ==================

target_nonre = nonre.select(
    nonre.BusinessPlanID.cast("Int"),
    nonre.DebtorCDI.cast("String"),
    nonre.AgreementNumber.cast("String"),
    nonre.BusinessPlan_Status.cast("String"),
    nonre.BusinessPlanStrategy_Code.cast("Int"),
    nonre.BusinessPlan_SolutionType.cast("String"),
    nonre.BusinessPlan_Comments.cast("String"),
    nonre.BusinessPlanStrategy_Status.cast("String"),
    nonre.Confidence.cast("String"),
    nonre.ProposedBy.cast("String"),
    nonre.BusinessPlanStrategy_StrategyType.cast("String"),
    nonre.BusinessPlanStrategy_Comments.cast("String"),
    nonre.LastModifiedDate.cast('Date'),
    nonre.PortfolioName.cast("String"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    nonre.NetRecoveryAmount.cast("Decimal(19, 2)"),
    nonre.BusinessDate.cast("Date").alias("BusinessDate"),
    nonre.TimeToRecovery.cast("Int").alias("TimeToRecovery"),
)

target_re_3rdparty = re_3rdparty.select(
    re_3rdparty.BusinessPlanID.cast("Int"),
    re_3rdparty.DebtorCDI.cast("String"),
    re_3rdparty.AgreementNumber.cast("String"),
    re_3rdparty.BusinessPlan_Status.cast("String"),
    re_3rdparty.BusinessPlanStrategy_Code.cast("Int"),
    re_3rdparty.BusinessPlan_SolutionType.cast("String"),
    re_3rdparty.BusinessPlan_Comments.cast("String"),
    re_3rdparty.BusinessPlanStrategy_Status.cast("String"),
    re_3rdparty.Confidence.cast("String"),
    re_3rdparty.ProposedBy.cast("String"),
    re_3rdparty.BusinessPlanStrategy_StrategyType.cast("String"),
    re_3rdparty.BusinessPlanStrategy_Comments.cast("String"),
    re_3rdparty.LastModifiedDate.cast('Date'),
    re_3rdparty.PortfolioName.cast("String"),
    re_3rdparty.TimeToAuction.cast("Int").alias("TimeToAuction"),
    re_3rdparty.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    re_3rdparty.NetRecoveryAmount.cast("Decimal(19, 2)"),
    re_3rdparty.BusinessDate.cast("Date").alias("BusinessDate"),
    lit(None).cast("Int").alias("TimeToRecovery"),
)

target_reo_hold = reo_hold.select(
    reo_hold.BusinessPlanID.cast("Int"),
    reo_hold.DebtorCDI.cast("String"),
    reo_hold.AgreementNumber.cast("String"),
    reo_hold.BusinessPlan_Status.cast("String"),
    reo_hold.BusinessPlanStrategy_Code.cast("Int"),
    reo_hold.BusinessPlan_SolutionType.cast("String"),
    reo_hold.BusinessPlan_Comments.cast("String"),
    reo_hold.BusinessPlanStrategy_Status.cast("String"),
    reo_hold.Confidence.cast("String"),
    reo_hold.ProposedBy.cast("String"),
    reo_hold.BusinessPlanStrategy_StrategyType.cast("String"),
    reo_hold.BusinessPlanStrategy_Comments.cast("String"),
    reo_hold.LastModifiedDate.cast('Date'),
    reo_hold.PortfolioName.cast("String"),
    reo_hold.TimeToAuction.cast("Int").alias("TimeToAuction"),
    reo_hold.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
    reo_hold.TimeToSell.cast("Int").alias("TimeToSell"),
    reo_hold.NetRecoveryAmount.cast("Decimal(19, 2)"),
    reo_hold.BusinessDate.cast("Date").alias("BusinessDate"),
    lit(None).cast("Int").alias("TimeToRecovery"),
)

target_reo_sale = reo_sale.select(
    reo_sale.BusinessPlanID.cast("Int"),
    reo_sale.DebtorCDI.cast("String"),
    reo_sale.AgreementNumber.cast("String"),
    reo_sale.BusinessPlan_Status.cast("String"),
    reo_sale.BusinessPlanStrategy_Code.cast("Int"),
    reo_sale.BusinessPlan_SolutionType.cast("String"),
    reo_sale.BusinessPlan_Comments.cast("String"),
    reo_sale.BusinessPlanStrategy_Status.cast("String"),
    reo_sale.Confidence.cast("String"),
    reo_sale.ProposedBy.cast("String"),
    reo_sale.BusinessPlanStrategy_StrategyType.cast("String"),
    reo_sale.BusinessPlanStrategy_Comments.cast("String"),
    reo_sale.LastModifiedDate.cast('Date'),
    reo_sale.PortfolioName.cast("String"),
    reo_sale.TimeToAuction.cast("Int").alias("TimeToAuction"),
    reo_sale.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
    reo_sale.TimeToSell.cast("Int").alias("TimeToSell"),
    reo_sale.NetRecoveryAmount.cast("Decimal(19, 2)"),
    reo_sale.BusinessDate.cast("Date").alias("BusinessDate"),
    lit(None).cast("Int").alias("TimeToRecovery"),
)

target_tool_forc_cbs = target_nonre.union(target_re_3rdparty).union(target_reo_hold).union(target_reo_sale)

# ================== Save results ==================
(
target_tool_forc_cbs
.write
.format("delta")
.clusterBy("BusinessDate")
.mode("append")
.saveAsTable(delivery_table_path)
)
print(delivery_table_path)